# Jamo test — 새로운 변형과 새로운 폰트
저장된 best.pt를 평가합니다. 학습이나 checkpoint 갱신은 하지 않습니다.

- seen_standard: 학습 폰트에서 새로운 seed로 학습 범위의 변형 생성
- seen_strong: 학습 폰트에서 더 큰 변형 생성
- unseen_standard / unseen_strong: 학습·검증 모두에 없던 폰트 평가

검증 폰트는 모델 선택에 사용됐으므로 true unseen test에서 제외합니다. 새 폰트가 없으면 unseen 평가만 건너뜁니다. 모든 reference/input은 동일 폰트입니다.


In [ ]:
from pathlib import Path
import sys, subprocess, tempfile, importlib
root = Path(tempfile.mkdtemp(prefix="jamo_test_code_", dir="/content"))
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/yechan25/hsqm.git", str(root)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(root / "requirements.txt")], check=True)
sys.path.insert(0, str(root))
for name in list(sys.modules):
    if name == "stroke_model" or name.startswith("stroke_model."):
        del sys.modules[name]
importlib.invalidate_caches()
commit = subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip()
from stroke_model.test_jamo import run_jamo_test
print("Test 코드:", commit)


## 설정
아래 checkpoint는 대화에서 확인한 학습 실행 경로입니다. 다른 실행의 best.pt를 평가하려면 CHECKPOINT만 바꾸세요.

새 폰트는 SOURCE_ROOT에 추가하거나, NEW_FONT_ROOT 아래에 폰트이름_unique_images/N_M_대표글자.png 구조로 준비하세요. 폰트 이름이 학습·검증 때와 같으면 새로운 폰트로 취급하지 않습니다.

TRAINING_ARCHIVE는 자동으로 식별합니다. 여러 ZIP이 후보이면 당시 학습에 쓴 jamo_*.zip 경로를 지정하세요. 생성된 test input이 기존 train/val input과 완전히 같은 경우는 제외합니다. 자모 조합 자체의 반복은 허용합니다.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CHECKPOINT = "/content/drive/Shareddrives/2026 자율연구/jamo seperation/jamo_training/jamo_run_20261014_153935_401612/best.pt"
SOURCE_ROOT = "/content/drive/Shareddrives/2026 자율연구/jamo seperation/unique_images"
NEW_FONT_ROOT = ""  # 선택: 새 폰트들의 *_unique_images 폴더가 들어 있는 상위 폴더
TRAINING_ARCHIVE = ""  # 자동 식별 실패 시 당시 학습 데이터 ZIP 경로
TEST_SEED = 20261015
SAMPLES_PER_CLASS = 30  # 폰트·클래스별 최대 test 조합; 빠른 확인은 5
PREVIEW_COUNT = 2  # 모드별 그림 개수 + 가장 낮은 Dice 샘플 1개
SOURCE_WORKERS = 8
BATCH_SIZE = 8
MAX_OVERLAP = 0.001


## Test 실행
standard: 크기 80~100%, 회전 ±5°, 이동 ±10px(128 기준).
strong: 크기 65~115%, 회전 ±15°, 이동 ±20px(128 기준).

강한 변형은 스트레스 테스트이며 모델의 모든 일반화를 대표하지 않습니다. 겹침/누락 등으로 제외된 조합과 수락 비율도 저장합니다. 애매한 교차 픽셀은 지표에서 제외되므로 annotation_coverage를 함께 확인하세요.


In [ ]:
summary, result_dir = run_jamo_test(
    CHECKPOINT, source_root=SOURCE_ROOT,
    new_font_root=NEW_FONT_ROOT or None,
    training_archive=TRAINING_ARCHIVE or None,
    seed=TEST_SEED, samples_per_class=SAMPLES_PER_CLASS,
    preview_count=PREVIEW_COUNT, workers=SOURCE_WORKERS,
    local_root="/content/jamo_test", batch_size=BATCH_SIZE, max_overlap=MAX_OVERLAP,
)
print("Checkpoint epoch:", summary["checkpoint_epoch"])
print("결과:", result_dir)


## 지표 비교
Dice/F1/IoU/Precision/Recall은 자모별 평균이며 높을수록 좋습니다. 이 평가에서 Dice와 F1은 같은 값입니다. Accuracy는 주석된 전경 픽셀 기준이며 배경은 포함하지 않습니다.

Reference baseline은 모델 없이 원본 자모의 가까운 위치로만 배정한 결과입니다. 모델 Dice와 baseline Dice의 차이도 함께 보세요. 폰트별·클래스별 지표와 개별 샘플 지표는 각 모드의 metrics.json에 있습니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
records = []
for name, result in summary["tests"].items():
    if result["status"] != "evaluated":
        print(name, "건너뜀:", result["reason"])
        continue
    scores = result["overall"]
    records.append({
        "Test": name,
        "Samples": scores["samples"],
        "Dice (%)": 100 * scores["macro_dice_annotated"],
        "F1 (%)": 100 * scores["macro_f1_annotated"],
        "IoU (%)": 100 * scores["macro_iou_annotated"],
        "Precision (%)": 100 * scores["macro_precision_annotated"],
        "Recall (%)": 100 * scores["macro_recall_annotated"],
        "FG Accuracy (%)": 100 * scores["foreground_accuracy_annotated"],
        "Coverage (%)": 100 * scores["annotation_coverage"],
        "Baseline Dice (%)": 100 * scores["reference_baseline_macro_dice"],
        "Accepted (%)": 100 * result["generation"]["acceptance_rate"],
    })
table = pd.DataFrame(records)
display(table.round(3))
table.to_csv(result_dir / "comparison.csv", index=False)
if not table.empty:
    ax = table.set_index("Test")[["Dice (%)", "Baseline Dice (%)"]].plot.bar(
        figsize=(9, 4), ylim=(0, 100), rot=15
    )
    ax.set_ylabel("Annotated macro Dice (%)")
    plt.tight_layout()
    plt.savefig(result_dir / "comparison.png", dpi=150)
    plt.show()
for name, result in summary["tests"].items():
    if result["status"] == "evaluated":
        print("\n", name, "폰트별 지표")
        display(pd.DataFrame.from_dict(result["by_group"]["font"], orient="index").round(5))
